# Lab B（fancy 版 · 實驗性）— 新的 GenAI Client SDK ＋ `.show()` 報告

> ⚠️ **實驗性、未實測**：這份用**新的** `vertexai.Client().evals`（GenAI Client in Vertex AI SDK），
> 它會渲染一個漂亮的 **Evaluation Report**（就是官方文件那種）。跟正式 Lab B（用舊的 `vertexai.evaluation`、給 pandas 表）並存、擇一。

差別一句話：**舊 SDK → pandas 表；新 SDK（這份）→ `.show()` 出 HTML 報告**。認證一樣用 Service Account。

## 0. 安裝（升級到有新 `vertexai.Client().evals` 的版本）

In [ ]:
!pip install -q -U "google-cloud-aiplatform[evaluation]"

## 1. 上傳 Service Account JSON（跟 Lab A/B 同一份）

In [ ]:
from google.colab import files
import os, json

print("請上傳 Service Account JSON 檔案：")
uploaded = files.upload()
sa_filename = os.path.abspath(list(uploaded.keys())[0])
with open(sa_filename) as f:
    sa_info = json.load(f)

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = sa_filename
PROJECT = sa_info["project_id"]
LOCATION = "us-central1"
print("ready (SA):", PROJECT)

## 2. 準備評估資料（保健品文案，第 3 筆故意寫壞）

In [ ]:
import pandas as pd

eval_df = pd.DataFrame({
    "prompt": [
        "幫魚油做台灣市場的行銷文案",
        "幫益生菌做台灣市場的行銷文案",
        "幫魚油做台灣市場的行銷文案",
    ],
    "response": [
        "【純淨深海魚油】每日補充 Omega-3，幫助調節生理機能，守護上班族的專注日常。",
        "【日常益生菌】幫助維持消化道機能，早晚一次，陪你養成順暢好習慣。",
        "【神效魚油】全台銷量第一！立即見效、根治高血壓，無副作用保證有效！",  # 故意違規
    ],
})
eval_df

## 3. 跑評估 ＋ `.show()` 出 fancy 報告

用新 SDK 的 `client.evals.evaluate(...)`，指標用**現成 rubric**（LLM-as-judge）：
- `TEXT_QUALITY`（品質）、`INSTRUCTION_FOLLOWING`（切題）、`SAFETY`（安全/合規向）。
跑完 `eval_result.show()` 會渲染 Summary Metrics ＋ 每個案例的評分與**理由**。

In [ ]:
import vertexai
from vertexai import types

client = vertexai.Client(project=PROJECT, location=LOCATION)

eval_result = client.evals.evaluate(
    dataset=eval_df,
    metrics=[
        types.RubricMetric.TEXT_QUALITY,
        types.RubricMetric.INSTRUCTION_FOLLOWING,
        types.RubricMetric.SAFETY,
    ],
)

eval_result.show()   # ← 就是這行渲染出漂亮的 Evaluation Report

## 收尾

- 這個漂亮報告來自**新的 `vertexai.Client().evals`**（GenAI Client SDK）＋ `.show()`。
- 正式 Lab B 用的是**舊的 `vertexai.evaluation`**（給 pandas 表，適合教學指著數字講）。
- **兩者都是 LLM-as-judge**，差別在 SDK 世代與呈現方式；核心觀念（judge 要驗證）不變。

> ⚠️ 這份實驗性、需在 Colab 實跑微調（欄位名 / 指標 / 版本可能要調）。